# DeepSORT — Appearance-Based Tracking

### Why SORT Fails

SORT matches objects using position (IoU) only:
- works well when objects are well separated
- fails when objects are close together
- fails when objects move fast
- fails during occlusion

Result in practice:
- 7 people in scene for 20 seconds
- SORT assigns 40+ unique IDs
- same person gets a new ID on every small movement

### DeepSORT — Adding Appearance

DeepSORT adds a second signal on top of position:

```
SORT      →  IoU only
DeepSORT  →  IoU  +  appearance similarity
```

Appearance pipeline:
- crop the detected object from the frame
- pass crop through a small CNN
- outputs an embedding (128 numbers)
- compare embeddings using cosine similarity

```
person crop  →  CNN  →  [0.2, 0.8, ...]  →  compare with previous embedding
```

<img src="images/cosine_similarity.png">

### Cosine Similarity — Quick Recap

Each embedding is an arrow pointing in some direction in 128D space:

```
person A  →  arrow pointing ↗
person B  →  arrow pointing →
same person A again  →  arrow pointing ↗  (same direction)
```

Cosine similarity measures the angle between two arrows:

| Angle | Similarity | Meaning |
| ----- | ---------- | ------- |
| 0° | 1.0 | identical |
| 45° | ~0.7 | somewhat similar |
| 90° | 0.0 | unrelated |
| 180° | -1.0 | totally opposite |

All embeddings are normalized to length 1 first:
- only direction matters, not length
- makes comparison fair regardless of crop size

### How DeepSORT Combines Both Signals

When matching a new box to a previous ID:

```
final_score = (IoU score × weight) + (appearance score × weight)
```

Not a strict AND condition — a weighted blend:
- object moved fast → IoU low, appearance carries the match
- long occlusion → IoU = 0, appearance still identifies the person
- normal movement → both signals contribute equally

**long occlusion**:
- a target object is hidden or obstructed by another object (an "occluder") for a long time / many frames

**short occulsion**:
- an object might be blocked for only a moment

This is why DeepSORT survives situations SORT cannot:

| Situation | SORT | DeepSORT |
| --------- | ---- | -------- |
| Normal movement | ✅ | ✅ |
| Fast movement | ❌ new ID | ✅ appearance matches |
| Two people cross | ❌ ID swap | ✅ appearance separates them |
| Short occlusion | ❌ ID lost | ✅ reappears with same ID |
| Long occlusion + same clothes | ❌ | ❌ still hard |

### BoT-SORT

ultralytics ships with BoT-SORT — a modern DeepSORT-style tracker:
- position matching via IoU
- appearance embeddings built in
- no external model needed

Switching from SORT to BoT-SORT:

```python
# SORT (default)
model.track(frame, persist=True)
# or
model.track(frame, tracker="bytetrack.yaml", persist=True)

# BoT-SORT
model.track(frame, tracker="botsort.yaml", persist=True)
```

One argument change — everything else stays identical.

### Setup

In [31]:
from ultralytics import YOLO
import cv2
import time

### Classes Color Sets

In [32]:
import random

random.seed(42)

COLORS = {
    class_id: tuple(random.randint(50, 255) for _ in range(3))
    for class_id in range(80)
}

### Constants

In [33]:
DETECT_EVERY   = 5
CONF_THRESHOLD = 0.5

### Load Model

In [ ]:
model = YOLO('yolov8n.pt')

### Tracking Function

In [35]:
def track(frame):

    results = model.track(frame, tracker="botsort.yaml", conf=CONF_THRESHOLD, persist=True, verbose=False, save=False)
    result  = results[0]

    detections = []

    for box in result.boxes:

        if box.id is None:
            continue

        class_id   = int(box.cls.item())
        label      = model.names[class_id]
        confidence = box.conf.item()
        track_id   = int(box.id.item())
        x1, y1, x2, y2 = map(int, box.xyxy[0].tolist())

        detections.append((label, class_id, confidence, track_id, (x1, y1, x2, y2)))

    return detections

#### `tracker="botsort.yaml"`
- switches from default ByteTrack to BoT-SORT
- enables appearance embedding matching
- yaml file is bundled inside ultralytics — no download needed

#### `save=False`
- disables ultralytics auto-saving results to `runs/detect/track`
- without this, ultralytics saves every frame to disk

### Draw Function

In [36]:
def draw(frame, detections):

    for label, class_id, confidence, track_id, (x1, y1, x2, y2) in detections:

        color = COLORS[class_id]
        
        cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)

        text = f"{label} {confidence:.2f} #{track_id}"
        cv2.putText(frame, text, (x1, y1 - 10),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, color, 2)

    return frame

### Webcam Loop

In [ ]:
cap = cv2.VideoCapture(0)

stored_detections = []
seen_ids          = set()
frame_count       = 0
fps               = 0
prev_time         = time.time()
display_fps       = "FPS: --"

while True:

    ret, frame = cap.read()
    if not ret:
        break

    frame_count += 1

    # Detection
    if frame_count % DETECT_EVERY == 0:

        t1 = time.time()
        stored_detections = track(frame)
        t2 = time.time()
        display_fps = f"FPS: {int(1 / (t2 - t1))}"
        
        for label, _, _, track_id, _ in stored_detections:
            if label == "person":
                seen_ids.add(track_id)
                
    if not stored_detections:
        display_fps = "Idle"

    # Draw
    frame = draw(frame, stored_detections)

    # Display
    cv2.putText(frame, display_fps, (20, 40),
                cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)

    cv2.putText(frame, f"Seen People: {len(seen_ids)}", (20, 80),
                cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)

    cv2.imshow("BoT-SORT Tracking", frame)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()

### SORT vs BoT-SORT — What To Expect

Run both and compare `Total seen` after the same amount of time:

```
Scene: 1 person, 20 seconds

SORT      →  Total seen: 30+   ❌
BoT-SORT  →  Total seen: 1-3   ✅
```

The closer `Total seen` is to the real number of people — the better the tracker.

Remaining failure cases even with BoT-SORT:
- very long occlusion (10+ seconds)
- identical appearance + identical position
- extreme lighting changes between frames

### Summary

| Concept | What It Means |
| ------- | ------------- |
| SORT | position-only matching via IoU |
| DeepSORT | position + appearance embedding matching |
| BoT-SORT | ultralytics built-in DeepSORT-style tracker |
| `tracker="botsort.yaml"` | switches from ByteTrack to BoT-SORT |
| `save=False` | disables auto-saving results to disk |
| Cosine similarity | measures angle between appearance embeddings |
| Weighted blend | IoU + appearance combined, not strict AND |

Next up — optical flow. Instead of detecting and tracking objects, we look at how pixels themselves move between frames.